# Project: Image Processing with NumPy

Implement core image processing operations using only NumPy array manipulations. Treat images as 3D arrays and apply transformations like grayscale, flipping, cropping, brightness adjustment, and edge detection.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print('NumPy version:', np.__version__)

## 1. Generate a Synthetic Image

In [ ]:
np.random.seed(42)
height, width = 200, 200
x = np.linspace(0, 1, width)
y = np.linspace(0, 1, height)
X, Y = np.meshgrid(x, y)
red = X
blue = Y
green = 1 - (X + Y) / 2
noise = np.random.normal(0, 0.05, (height, width))
image = np.stack([red, green, blue], axis=-1)
image = np.clip(image + noise[..., np.newaxis], 0, 1)
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title('Original Synthetic Image')
plt.axis('off')
plt.show()
print('Image shape:', image.shape, '| dtype:', image.dtype)

## 2. Grayscale Conversion

Weights: 0.2989 R + 0.5870 G + 0.1140 B

In [ ]:
weights = np.array([0.2989, 0.5870, 0.1140])
grayscale = image @ weights
plt.figure(figsize=(6, 6))
plt.imshow(grayscale, cmap='gray')
plt.title('Grayscale')
plt.axis('off')
plt.show()
print('Grayscale shape:', grayscale.shape)

## 3. Image Flipping and Rotation

In [ ]:
flipped_h = image[:, ::-1, :]
flipped_v = image[::-1, :, :]
rotated = np.rot90(image)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(flipped_h)
axes[0].set_title('Horizontal Flip')
axes[0].axis('off')
axes[1].imshow(flipped_v)
axes[1].set_title('Vertical Flip')
axes[1].axis('off')
axes[2].imshow(rotated)
axes[2].set_title('Rotated 90 deg')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 4. Cropping and Resizing

In [ ]:
crop = image[50:150, 50:150, :]

def downsample(img, factor=2):
    h, w = img.shape[:2]
    h2, w2 = h // factor, w // factor
    return img[:h2*factor, :w2*factor].reshape(h2, factor, w2, factor, -1).mean(axis=(1, 3))

downsampled = downsample(image, factor=4)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(crop)
axes[0].set_title('Cropped (100x100)')
axes[0].axis('off')
axes[1].imshow(np.clip(downsampled, 0, 1))
axes[1].set_title(f'Downsampled {downsampled.shape[:2]}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 5. Brightness and Contrast

In [ ]:
bright = np.clip(image + 0.3, 0, 1)
dark = np.clip(image - 0.3, 0, 1)
high_contrast = np.clip((image - 0.5) * 1.8 + 0.5, 0, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(bright)
axes[0].set_title('Brightness +0.3')
axes[0].axis('off')
axes[1].imshow(dark)
axes[1].set_title('Brightness -0.3')
axes[1].axis('off')
axes[2].imshow(high_contrast)
axes[2].set_title('Contrast x1.8')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 6. Edge Detection (Sobel Operator)

In [ ]:
def sobel_edge_detection(img):
    if img.ndim == 3:
        img = img @ np.array([0.2989, 0.5870, 0.1140])
    Gx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    Gy = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])
    def convolve2d(img, kernel):
        h, w = img.shape
        kh, kw = kernel.shape
        pad_h, pad_w = kh // 2, kw // 2
        padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), 'edge')
        result = np.zeros_like(img)
        for i in range(h):
            for j in range(w):
                result[i, j] = np.sum(padded[i:i+kh, j:j+kw] * kernel)
        return result
    grad_x = convolve2d(img, Gx)
    grad_y = convolve2d(img, Gy)
    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    return magnitude / magnitude.max()

edges = sobel_edge_detection(image)
plt.figure(figsize=(6, 6))
plt.imshow(edges, cmap='gray')
plt.title('Edge Detection (Sobel)')
plt.axis('off')
plt.show()

## Summary

- Images are 3D NumPy arrays (H x W x C)
- Slicing for crop and flip operations
- Vectorized operations for brightness/contrast
- Custom convolution for edge detection